# Rapido Captain Acquisition & Supply Optimization

This notebook analyzes the captain onboarding funnel, campaign performance, and airport demand-supply gaps using the synthetic data provided for the assignment.

The main objective is to identify where captain supply is being lost and which interventions are most likely to increase productive captain supply.

In [ ]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt


# Find the project root reliably, whether the notebook is
# opened from VS Code or executed with nbconvert.
current_path = Path.cwd()

if (current_path / "run_analysis.py").exists():
    PROJECT_ROOT = current_path
elif (current_path.parent / "run_analysis.py").exists():
    PROJECT_ROOT = current_path.parent
else:
    PROJECT_ROOT = next(
        parent
        for parent in [current_path] + list(current_path.parents)
        if (parent / "run_analysis.py").exists()
    )


DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
RESULTS_DIR = OUTPUT_DIR / "results"
TABLES_DIR = OUTPUT_DIR / "tables"

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Project root :", PROJECT_ROOT)
print("Data folder  :", DATA_DIR)
print("Results      :", RESULTS_DIR)
print("Tables       :", TABLES_DIR)

## 1. Data overview

The raw data contains captain signup information, document events, approval and activation outcomes, campaign/nudge information, and airport demand-supply and trip data.

In [ ]:
raw_files = [
    "captains.csv",
    "doc_events.csv",
    "approvals.csv",
    "activation.csv",
    "nudges.csv",
    "airport_hourly.csv",
    "airport_trips.csv",
]

raw_summary = []

for filename in raw_files:
    path = DATA_DIR / filename

    if path.exists():
        df = pd.read_csv(path)
        raw_summary.append(
            {
                "dataset": filename,
                "rows": len(df),
                "columns": len(df.columns),
            }
        )
    else:
        raw_summary.append(
            {
                "dataset": filename,
                "rows": None,
                "columns": None,
            }
        )

raw_summary = pd.DataFrame(raw_summary)

raw_summary

## 2. Data quality

The production pipeline performs validation before analysis.

The latest pipeline run completed the validation step with zero reported data-quality issues.

In [ ]:
quality_path = RESULTS_DIR / "data_quality_report.json"

with open(quality_path, "r", encoding="utf-8") as file:
    quality_report = json.load(file)

print("Datasets checked :", quality_report.get("datasets_checked"))
print("Total issues     :", quality_report.get("total_issues"))

if quality_report.get("issues"):
    display(pd.DataFrame(quality_report["issues"]))
else:
    print("No data quality issues detected.")

## 3. A1 — Captain onboarding funnel

The funnel follows the document sequence specified in the assignment:

DL → RC → Aadhaar → Permit → Fitness → Insurance

Permit is required for Auto and Cab. ERickshaw does not require a Permit.

The final outcome is extended beyond approval to first-order completion because approval alone does not guarantee productive supply.

In [ ]:
funnel_path = TABLES_DIR / "onboarding_funnel.csv"

funnel = pd.read_csv(funnel_path)

funnel

## 4. A1 — Captain onboarding funnel

The funnel follows the document order given in the assignment:

DL → RC → Aadhaar → Permit → Fitness → Insurance → Approval → First Order

The main outcome is not only approval. I also track first-order completion because an approved captain does not necessarily become productive supply.

In [ ]:
funnel_path = TABLES_DIR / "onboarding_funnel.csv"

if not funnel_path.exists():
    raise FileNotFoundError(
        f"Funnel output not found: {funnel_path}\n"
        "Run `python run_analysis.py` before executing the notebook."
    )

funnel = pd.read_csv(funnel_path)

print("Onboarding funnel:")
display(funnel)

In [ ]:
if {"stage", "captains"}.issubset(funnel.columns):

    plt.figure(figsize=(11, 5))

    plt.plot(
        funnel["stage"],
        funnel["captains"],
        marker="o"
    )

    plt.title("Captain Onboarding Funnel")
    plt.xlabel("Stage")
    plt.ylabel("Captains")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

else:
    print("Available columns:")
    print(funnel.columns.tolist())

### Funnel takeaway

Only 6.4% of signups reach a completed first order.

The largest absolute document-stage loss occurs at Insurance, where 3,577 captains are lost between Fitness and Insurance clearance.

The largest post-approval loss occurs between Approval and First Order:

4,206 approved → 1,610 first orders

This leaves 2,596 approved captains who do not complete a first order in the observed data.

## 5. A2 — Actionable onboarding segments

I look at document leakage by captain segment rather than only looking at the overall funnel.

The segmentation uses city, vehicle type, acquisition channel, device tier, app language, and age band where available.

For prioritization, I focus on the number of captains who have not cleared a required document and estimate how much of that gap could realistically be recovered.

In [ ]:
segment_path = TABLES_DIR / "a2_actionable_segments.csv"

segment_leaks = pd.read_csv(segment_path)

print("Rows:", len(segment_leaks))
print("Columns:")
print(segment_leaks.columns.tolist())

segment_leaks.head(15)

In [ ]:
if "actionability" in segment_leaks.columns:

    prioritized_segments = segment_leaks.sort_values(
        "actionability",
        ascending=False
    )

elif "not_cleared" in segment_leaks.columns:

    prioritized_segments = segment_leaks.sort_values(
        "not_cleared",
        ascending=False
    )

else:

    prioritized_segments = segment_leaks.copy()

prioritized_segments.head(15)

### A2 takeaway

Permit completion is the strongest actionable segment-level opportunity among Auto/Cab captains.

The highest-priority segments include Pune Auto and Hyderabad Auto/Cab, particularly organic and referral acquisition segments.

For example, Pune Auto + organic_app at the Permit stage has 744 captains who did not clear the stage. Under a 20% recovery scenario, this represents approximately 149 potentially recoverable approvals and around 57 expected first orders at the observed 38.3% approval-to-first-order rate.

These are scenario estimates and should not be interpreted as causal forecasts.

## 6. A3 — CAMP_WA_002

The campaign is evaluated by comparing approval outcomes for treated and control captains.

Because campaign exposure was not randomized, the adjusted result is treated as an observational association rather than a causal estimate.

In [ ]:
campaign_path = RESULTS_DIR / "campaign_confidence.json"

with open(campaign_path, "r", encoding="utf-8") as file:
    campaign = json.load(file)

campaign_summary = pd.DataFrame(
    [
        {
            "treated_captains": campaign["treated_captains"],
            "control_captains": campaign["control_captains"],
            "treated_approval_rate": campaign["treated_approval_rate"],
            "control_approval_rate": campaign["control_approval_rate"],
            "observed_lift_pp": campaign["observed_lift_pp"],
            "adjusted_lift_pp": campaign["adjusted_lift_pp"],
            "odds_ratio": campaign["odds_ratio"],
            "p_value": campaign["p_value"],
        }
    ]
)

campaign_summary

### A3 takeaway

CAMP_WA_002 is associated with a +16.7 percentage point adjusted approval lift.

The observed lift is +17.9 percentage points with a 95% confidence interval of approximately +16.8 to +19.0 percentage points.

The association is strong, but the campaign was not randomized. I would therefore run a randomized holdout before scaling the campaign broadly.

## 7. B1/B2 — Airport demand and post-trip behavior

The airport analysis looks at both sides of the problem:

1. Where demand is not being fulfilled.
2. What happens after airport trips.

This helps distinguish a general acquisition problem from a time- and location-specific supply problem.

In [ ]:
airport_zone = pd.read_csv(
    TABLES_DIR / "airport_zone_summary.csv"
)

airport_hourly = pd.read_csv(
    TABLES_DIR / "airport_hourly_summary.csv"
)

print("Airport zone summary")
display(airport_zone.head(10))

print("Airport hourly summary")
display(airport_hourly.head(10))

In [ ]:
airport_gap = pd.read_csv(
    TABLES_DIR / "b3_airport_supply_gap.csv"
)

print("Columns:")
print(airport_gap.columns.tolist())

airport_gap.head(15)

In [ ]:
if {"hour", "fulfillment_rate"}.issubset(airport_hourly.columns):

    hourly_plot = airport_hourly.sort_values("hour")

    plt.figure(figsize=(11, 5))

    plt.plot(
        hourly_plot["hour"],
        hourly_plot["fulfillment_rate"],
        marker="o"
    )

    plt.title("Airport Fulfillment Rate by Hour")
    plt.xlabel("Hour of Day")
    plt.ylabel("Fulfillment Rate")
    plt.xticks(range(24))
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

else:
    print("The hourly table does not contain the expected columns.")

### B1/B2 takeaway

Airport supply pressure is concentrated in the late-night period, particularly around 21:00–03:00.

Late-night airport trips also have a higher cancellation rate and a lower return-fare-within-20-min rate than the overall airport-trip population.

This suggests that the supply issue is not evenly distributed across the day and should be addressed with targeted interventions rather than broad captain acquisition.

In [ ]:
trip_overall_path = TABLES_DIR / "b2_airport_trip_overall.csv"
trip_hourly_path = TABLES_DIR / "b2_airport_trip_hourly.csv"

trip_overall = pd.read_csv(trip_overall_path)
trip_hourly = pd.read_csv(trip_hourly_path)

print("Overall airport trip behavior")
display(trip_overall)

print("Hourly airport trip behavior")
display(trip_hourly.head(24))

### B2 takeaway

Overall airport trips have:

- 13.77% cancellation rate
- 35.96% return fare within 20 minutes

For the late-night 21:00–03:00 window:

- Cancellation rate is approximately 17.09%
- Return fare within 20 minutes is approximately 29.84%

The late-night period therefore combines weaker fulfillment with higher cancellation and lower short-window return-trip behavior.

## 8. B3 — Should Rapido acquire more airport captains?

The analysis does not recommend immediately acquiring a large number of captains.

The preferred sequence is:

1. Target the late-night airport shortage.
2. Use incentives and repositioning to improve existing supply.
3. Measure whether fulfillment improves.
4. If the gap persists, acquire captains specifically for the affected airport/time/vehicle segments.

This reduces the risk of adding supply during periods when it is not needed.

In [ ]:
b3_path = TABLES_DIR / "b3_airport_recommendation.csv"

b3_recommendation = pd.read_csv(b3_path)

b3_recommendation

## 9. Decision framework

The final recommendation is based on productive supply rather than signup volume alone.

For onboarding interventions:

Recoverable captains
× downstream approval probability
× approval-to-first-order rate
= expected incremental first orders

The current scenario assumes a 20% recovery rate and uses the observed 38.3% approval-to-first-order rate.

For airport supply, the priority is:

Demand gap
→ existing-supply intervention
→ controlled measurement
→ targeted acquisition if the gap remains

## 10. Final recommendations

### 1. Prioritize Permit completion

Focus on high-leakage Auto/Cab segments, especially Pune and Hyderabad.

A 20% recovery scenario corresponds to approximately 397 incremental Permit-stage approvals per month across the analyzed mature segment-month population.

### 2. Validate CAMP_WA_002

The campaign shows a +16.7 pp adjusted association with approval, but a randomized holdout is required before treating this as causal impact.

### 3. Improve Approved → First Order

2,596 approved captains do not complete a first order. Improving activation can create productive supply without acquiring another captain.

### 4. Target airport supply interventions

Focus on the 21:00–03:00 shortage window using incentives and repositioning before investing in targeted acquisition.

## 11. Limitations

- The data is synthetic and represents the assignment's supplied dataset.
- Intervention impact is scenario-based and not a causal estimate.
- CAMP_WA_002 exposure was not randomized.
- First-order completion is used as the available proxy for productive captain activation.
- The airport supply gap should not be interpreted as a direct requirement to acquire that exact number of captains.
- Long-term captain productivity, retention, and contribution margin are not available in the supplied data.